# Day 83: Embedding Similarity Search Pipeline

**Layer:** Modalities


## Concept Overview

An embedding search pipeline has three stages: embed (model inference), index (FAISS/HNSW), and query (ANN search). The bottleneck is usually embedding throughput at index build time.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Similarity Search Pipeline

Build a toy FAISS-like index and measure embed -> index -> query latency.


In [ ]:
import numpy as np, time

class VectorIndex:
    """Simple flat L2 index."""
    def __init__(self, dim):
        self.dim = dim
        self.vectors = []
        self.ids = []

    def add(self, vecs, ids):
        self.vectors.extend(vecs)
        self.ids.extend(ids)

    def search(self, query, k=5):
        if not self.vectors: return [], []
        vecs = np.array(self.vectors)
        dists = np.sum((vecs - query)**2, axis=1)
        top_k = np.argsort(dists)[:k]
        return [self.ids[i] for i in top_k], dists[top_k]

np.random.seed(42)
dim = 768
n_docs = 10000

# Simulate embedding
t0 = time.perf_counter()
vectors = np.random.randn(n_docs, dim).astype(np.float32)
embed_ms = (time.perf_counter()-t0)*1000

# Index
t0 = time.perf_counter()
idx = VectorIndex(dim)
idx.add(vectors, list(range(n_docs)))
index_ms = (time.perf_counter()-t0)*1000

# Query
query = np.random.randn(dim).astype(np.float32)
t0 = time.perf_counter()
for _ in range(100):
    ids, dists = idx.search(query)
query_ms = (time.perf_counter()-t0)*10  # avg ms

print(f"Pipeline: {n_docs} docs, dim={dim}")
print(f"  Embed time:  {embed_ms:.1f}ms ({n_docs/(embed_ms/1000):.0f} vecs/s)")
print(f"  Index build: {index_ms:.1f}ms")
print(f"  Query (flat L2): {query_ms:.2f}ms avg")
print()
print("Production: FAISS HNSW reduces query from O(N) to O(log N).")


## Experiments: Try These

1. Extend the implementation with an additional feature.
2. Benchmark on real hardware and compare to theoretical estimates.
3. Connect this to a previous day's implementation.


## Key Takeaways

- An embedding search pipeline has three stages: embed (model inference), index (FAISS/HNSW), and query (ANN search).
- An embedding search pipeline has three stages: embed (model inference), index (F....
- Day 83 implementation complete.
